# Data exploration

In [1]:
from datetime import datetime
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split


CSV_FILE_TRAIN = "data/train.csv"
CSV_FILE_TEST = "data/test.csv"

df = pd.read_csv(CSV_FILE_TRAIN)
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


## Is Sex a factor in the survival rate ?

In [2]:
#  Sex statistics
missing = df['Sex'].isnull().sum()
print(f"Number of null value is {missing}")

Number of null value is 0


In [3]:
# Sex distribution
df.groupby('Sex')['PassengerId'].count()

Sex
female    314
male      577
Name: PassengerId, dtype: int64

In [4]:
# Survival rate per sex
df.groupby('Sex')['Survived'].mean()

Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64

Conclusion: females have a 74% survival rate, while males have a 18% survival rate.

## Is socio-economic status (through PClass column) a factor ?

In [5]:
#  Pclass statistics
missing = df['Pclass'].isna().sum()
mean = df['Pclass'].mean()
std = df['Pclass'].std()
print(f"Number of null value is {missing}")
print(f"Mean is {mean}")
print(f"Standard deviation is {std}")

Number of null value is 0
Mean is 2.308641975308642
Standard deviation is 0.836071240977049


In [6]:
# Pclass distribution
df.groupby('Pclass')['PassengerId'].count()

Pclass
1    216
2    184
3    491
Name: PassengerId, dtype: int64

In [7]:
# Survival rate per Pclass
df.groupby('Pclass')['Survived'].mean()

Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64

Conclusion: 1st class has a 62% survival rate while 3rd class has a 24% survival rate

## Is age a factor in the survival rate ?

In [8]:
#  Age statistics
missing = df['Age'].isna().sum()
mean = df['Age'].mean()
std = df['Age'].std()
print(f"Number of missing value is {missing}")
print(f"Mean is {mean}")
print(f"Standard deviation is {std}")

Number of missing value is 177
Mean is 29.69911764705882
Standard deviation is 14.526497332334042


In [9]:
# Age group distribution
def get_age_group(age):
    if age <= 2:
        return '00-02'
    elif age <= 6:
        return '02-06'
    elif age <= 12:
        return '06-12'
    elif age <= 20:
        return '12-20'
    elif age <= 30:
        return '20-30'
    elif age <= 40:
        return '30-40'
    elif age <= 50:
        return '40-50'
    elif age <= 60:
        return '50-60'
    elif age <= 70:
        return '60-70'
    else:
        return '70+'


# Remove records where age is missing
df_with_age = df[df['Age'].notna()]

# Create new 'age_group' column
df['age_group'] = df_with_age['Age'].apply(get_age_group)
df.groupby('age_group')['PassengerId'].count()

age_group
00-02     24
02-06     23
06-12     22
12-20    110
20-30    230
30-40    155
40-50     86
50-60     42
60-70     17
70+        5
Name: PassengerId, dtype: int64

In [10]:
# Survival rate per age group
df.groupby('age_group')['Survived'].mean()

age_group
00-02    0.625000
02-06    0.782609
06-12    0.318182
12-20    0.381818
20-30    0.365217
30-40    0.445161
40-50    0.383721
50-60    0.404762
60-70    0.235294
70+      0.200000
Name: Survived, dtype: float64

Conclusions:
    - children (< 6yo) are more likely to survive
    - elderly people > 60yo have are less likely to survive
    - 10-20 yo have a similar survival rate as 20-60

As a first iteration, a small model could be created with the following features: sex, pclass, age group.

## Dealing with missing age information
- Age is missing for a lot of entries (177 out of 891, i.e. ~20%)
- We will extrapolate age based on the 'Name' column because it usually contains a title ('Miss', 'Mrs', 'Mr', 'Master, ...)
- The title is used as a proxy for age. We will assign the mean age for that particular title.

In [11]:
# Exploring names for children (12 yo) to find patterns in their names
filtered_df = df[df['Age'] < 12]
filtered_df[['Name', 'Age', 'Sex']].sort_values(by='Age', ascending=True).tail(50)

,Name,Age,Sex
297,"Allison, Miss. Helen Loraine",2.0,female
205,"Strom, Miss. Telma Matilda",2.0,female
530,"Quick, Miss. Phyllis May",2.0,female
479,"Hirvonen, Miss. Hildur E",2.0,female
119,"Andersson, Miss. Ellis Anna Maria",2.0,female
340,"Navratil, Master. Edmond Roger",2.0,male
261,"Asplund, Master. Edvin Rojj Felix",3.0,male
193,"Navratil, Master. Michel M",3.0,male
348,"Coutts, Master. William Loch ""William""",3.0,male
374,"Palsson, Miss. Stina Viola",3.0,female


Conclusion:
    - Almost all female children have 'Miss.' in their name
    - Almost all male children have 'Master.' in their name
    - Starting age 11, some male are called 'Mr.'

In [12]:
# Extract title from Name column
def get_name_title(name):
    if any(t in name for t in ['Miss.', 'Mlle.']):
        return 'Miss'
    elif 'Master.' in name:
        return 'Master'
    if any(t in name for t in ['Mrs.', 'Mme.']):
        return 'Mrs'
    if any(t in name for t in ['Mr.', 'Dr.', 'Rev']):
        return 'Mr'
    else:
        return 'Other'


# Create new 'age_group' column
df['title'] = df['Name'].apply(get_name_title)
df.groupby('title')['PassengerId'].count()

title
Master     40
Miss      184
Mr        530
Mrs       126
Other      11
Name: PassengerId, dtype: int64

In [13]:
# Explore names without titles
df_no_title = df[df['title'] == 'Other']
df_no_title[['Name', 'Age']].sort_values(by='Age')

,Name,Age
443,"Reynaldo, Ms. Encarnacion",28.0
759,"Rothes, the Countess. of (Lucy Noel Martha Dye...",33.0
822,"Reuchlin, Jonkheer. John George",38.0
30,"Uruchurtu, Don. Manuel E",40.0
536,"Butt, Major. Archibald Willingham",45.0
556,"Duff Gordon, Lady. (Lucille Christiana Sutherl...",48.0
599,"Duff Gordon, Sir. Cosmo Edmund (""Mr Morgan"")",49.0
449,"Peuchen, Major. Arthur Godfrey",52.0
647,"Simonius-Blumer, Col. Oberst Alfons",56.0
694,"Weir, Col. John",60.0


Conclusion:
- those 11 passengers are considered edge cases, and age is provided for all of them anyways

In [14]:
# Compute median age by title
df_median_title_age = df.groupby('title', as_index=False)['Age'].median()
df_median_title_age

,title,Age
0,Master,3.5
1,Miss,21.0
2,Mr,30.0
3,Mrs,35.0
4,Other,48.0


In [15]:
# Creating dictionary mapping titles to median ages
title_age_map = dict(zip(df_median_title_age['title'], df_median_title_age['Age']))
title_age_map

{'Master': 3.5, 'Miss': 21.0, 'Mr': 30.0, 'Mrs': 35.0, 'Other': 48.0}

In [16]:
filtered_df = df[df['Name'].str.contains('Miss.', case=False, na=False)].dropna(subset=['Age'])
filtered_df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,age_group,title
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,20-30,Miss
10,11,1,3,"Sandstrom, Miss. Marguerite Rut",female,4.0,1,1,PP 9549,16.7000,G6,S,02-06,Miss
11,12,1,1,"Bonnell, Miss. Elizabeth",female,58.0,0,0,113783,26.5500,C103,S,50-60,Miss
14,15,0,3,"Vestrom, Miss. Hulda Amanda Adolfina",female,14.0,0,0,350406,7.8542,NaN,S,12-20,Miss
22,23,1,3,"McGowan, Miss. Anna ""Annie""",female,15.0,0,0,330923,8.0292,NaN,Q,12-20,Miss
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
853,854,1,1,"Lines, Miss. Mary Conover",female,16.0,0,1,PC 17592,39.4000,D28,S,12-20,Miss
866,867,1,2,"Duran y More, Miss. Asuncion",female,27.0,1,0,SC/PARIS 2149,13.8583,NaN,C,20-30,Miss
875,876,1,3,"Najib, Miss. Adele Kiamie ""Jane""",female,15.0,0,0,2667,7.2250,NaN,C,12-20,Miss
882,883,0,3,"Dahlberg, Miss. Gerda Ulrika",female,22.0,0,0,7552,10.5167,NaN,S,20-30,Miss


# Data preparation

## Normalizing data before transforming it into a pytorch tensor

In [17]:
# Create buckets for Age
def get_age_group_numeric(age):
    if age <= 2:
        return 0.0
    elif age <= 6:
        return 1.0
    elif age <= 12:
        return 2.0
    elif age <= 20:
        return 3.0
    elif age <= 30:
        return 4.0
    elif age <= 40:
        return 5.0
    elif age <= 50:
        return 6.0
    elif age <= 60:
        return 7.0
    elif age <= 70:
        return 8.0
    else:
        return 9.0

In [18]:
# Normalize input data
# x: pandas dataframe containing the Titanic dataset
def normalize(df):
    # Set Sex=1 for female and Sex=0 for male
    df['Sex'] = df['Sex'].map({'female': 1, 'male': 0})

    # Extract title from 'Name' column
    df['Title'] = df['Name'].apply(get_name_title)

    # Filling missing Age with median age for corresponding title
    df['Age'] = df['Age'].fillna(df.apply(lambda row: title_age_map[row['Title']], axis=1))

    # Creating age buckets
    df['AgeGroup'] = df['Age'].apply(get_age_group_numeric)

    return df

In [19]:
normalize(df).head(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,age_group,title,Title,AgeGroup
0,1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,NaN,S,20-30,Mr,Mr,4.0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,C85,C,30-40,Mrs,Mrs,5.0
2,3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,20-30,Miss,Miss,4.0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,C123,S,30-40,Mrs,Mrs,5.0
4,5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,NaN,S,30-40,Mr,Mr,5.0
5,6,0,3,"Moran, Mr. James",0,30.0,0,0,330877,8.4583,NaN,Q,NaN,Mr,Mr,4.0
6,7,0,1,"McCarthy, Mr. Timothy J",0,54.0,0,0,17463,51.8625,E46,S,50-60,Mr,Mr,7.0
7,8,0,3,"Palsson, Master. Gosta Leonard",0,2.0,3,1,349909,21.0750,NaN,S,00-02,Master,Master,0.0
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",1,27.0,0,2,347742,11.1333,NaN,S,20-30,Mrs,Mrs,4.0
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",1,14.0,1,0,237736,30.0708,NaN,C,12-20,Mrs,Mrs,3.0


## Load data in Pytorch datasets

In [20]:
class TitanicDataset(Dataset):
    def __init__(self, csv_file, features_cols, label_col=None):
        """
        Initialize Dataset from CSV file
        """
        df = pd.read_csv(csv_file)
        df = normalize(df)

        self.features_cols = features_cols
        self.label_col = label_col
        self.has_labels = label_col is not None

        # Convert data into pytorch tensors
        self.features = torch.tensor(df[features_cols].values, dtype=torch.float32)
        if self.has_labels:
            self.labels = torch.tensor(df[self.label_col].values, dtype=torch.long)

    def __len__(self):
        """
        Retourne le nombre d'échantillons.
        """
        return len(self.features)

    def __getitem__(self, index):
        """
        Retourne un échantillon (features, label) à l'index donné.
        """
        if self.has_labels:
            return self.features[index], self.labels[index]
        else:
            return self.features[index]


features_cols = ['Sex', 'Pclass', 'AgeGroup']
label_col = 'Survived'

# Use 80% of the dataset to train our model
# Remaning 20% are used to validate the model with unseen data (validation dataset)
train_dataset_length_pct = 0.8
val_dataset_length_pct = 1 - train_dataset_length_pct

dataset = TitanicDataset(CSV_FILE_TRAIN, features_cols, label_col)
train_dataset, val_dataset = random_split(dataset, lengths=[train_dataset_length_pct, val_dataset_length_pct])

# Model training

## Create neural network

In [21]:
# Get cpu, gpu or mps device for training.
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")


# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_relu_stack = nn.Sequential(
            #nn.Linear(2, 4),
            #nn.ReLU(),
            nn.Linear(3, 2)
        )

    def forward(self, x):
        logits = self.linear_relu_stack(x)
        return logits


model = NeuralNetwork().to(device)
print(model)

Using cpu device
NeuralNetwork(
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=3, out_features=2, bias=True)
  )
)


In [22]:
batch_size = 16
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [23]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 8 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [24]:
learning_rate = 1e-2
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

epochs = 50
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(val_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 1.636976  [   16/  713]
loss: 0.780171  [  144/  713]
loss: 0.677136  [  272/  713]
loss: 0.674402  [  400/  713]
loss: 0.594785  [  528/  713]
loss: 0.803051  [  656/  713]
Test Error: 
 Accuracy: 61.8%, Avg loss: 0.658626 

Epoch 2
-------------------------------
loss: 0.643163  [   16/  713]
loss: 0.663980  [  144/  713]
loss: 0.619271  [  272/  713]
loss: 0.588866  [  400/  713]
loss: 0.551754  [  528/  713]
loss: 0.780936  [  656/  713]
Test Error: 
 Accuracy: 64.0%, Avg loss: 0.629151 

Epoch 3
-------------------------------
loss: 0.599334  [   16/  713]
loss: 0.673978  [  144/  713]
loss: 0.600644  [  272/  713]
loss: 0.532214  [  400/  713]
loss: 0.524744  [  528/  713]
loss: 0.763032  [  656/  713]
Test Error: 
 Accuracy: 67.4%, Avg loss: 0.608988 

Epoch 4
-------------------------------
loss: 0.566698  [   16/  713]
loss: 0.682909  [  144/  713]
loss: 0.586061  [  272/  713]
loss: 0.489910  [  400/  713]
loss: 0.503851  [  528/ 

# Analyze model parameters

In [27]:
for param in model.parameters():
    print(param)

Parameter containing:
tensor([[-0.7430,  0.8449,  0.3716],
        [ 1.5728, -0.0272,  0.1981]], requires_grad=True)
Parameter containing:
tensor([-0.6926,  0.5635], requires_grad=True)


In [35]:
# Logits for different entries
for sex, sex_encoded in [('male', 0.0), ('female', 1.0)]:
    for pclass in [1.0, 2.0, 3.0]:
        for age_group, age_group_encoded in [('02-06', 1.0), ('06-12', 2.0), ('20-30', 4.0), ('60-70', 8.0)]:
            pred = model(torch.tensor([sex_encoded, pclass, age_group_encoded])).tolist()
            print(f"Sex={sex}, Pclass={pclass}, AgeGroup={age_group} : {pred}")

Sex=male, Pclass=1.0, AgeGroup=02-06 : [0.5239879488945007, 0.7343294620513916]
Sex=male, Pclass=1.0, AgeGroup=06-12 : [0.8956167101860046, 0.9323844909667969]
Sex=male, Pclass=1.0, AgeGroup=20-30 : [1.6388745307922363, 1.3284947872161865]
Sex=male, Pclass=1.0, AgeGroup=60-70 : [3.125389575958252, 2.1207151412963867]
Sex=male, Pclass=2.0, AgeGroup=02-06 : [1.3689234256744385, 0.7071273922920227]
Sex=male, Pclass=2.0, AgeGroup=06-12 : [1.7405524253845215, 0.9051824808120728]
Sex=male, Pclass=2.0, AgeGroup=20-30 : [2.4838099479675293, 1.3012926578521729]
Sex=male, Pclass=2.0, AgeGroup=60-70 : [3.970325469970703, 2.093513011932373]
Sex=male, Pclass=3.0, AgeGroup=02-06 : [2.2138590812683105, 0.6799253225326538]
Sex=male, Pclass=3.0, AgeGroup=06-12 : [2.5854878425598145, 0.8779804706573486]
Sex=male, Pclass=3.0, AgeGroup=20-30 : [3.3287458419799805, 1.2740905284881592]
Sex=male, Pclass=3.0, AgeGroup=60-70 : [4.815260887145996, 2.0663108825683594]
Sex=female, Pclass=1.0, AgeGroup=02-06 : [-0

# Predictions on test set and submission

In [36]:
def predict(dataloader, model):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    # Initiliaze empty tensor that will contain our predictions
    pred = torch.tensor([])
    with torch.no_grad():
        for X in dataloader:
            logits = model(X)
            batch_pred = logits.argmax(1)
            pred = torch.cat((pred, batch_pred))
    return pred

In [37]:
# Write CSV file ready to be submitted to Kaggle
# pred: pytorch tensor (m, 1) with predictions
def write_submission_file(csv_file_test, pred):
    df = pd.read_csv(csv_file_test)
    df['Survived'] = pred.tolist()
    df['Survived'] = df['Survived'].astype(int)
    df = df[['PassengerId', 'Survived']]

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"tbriot_submission_{timestamp}.csv"
    df.to_csv(filename, index=False)
    print(f"Submission {filename} has been written!")

In [38]:
features_cols = ['Sex', 'Pclass', 'AgeGroup']
test_batch_size = 64

test_dataset = TitanicDataset(CSV_FILE_TEST, features_cols)
test_dataloader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False)

# Invoke model to get predictions
pred = predict(test_dataloader, model)
# Write predictions to submission file
write_submission_file(CSV_FILE_TEST, pred)

Submission tbriot_submission_20250113_164747.csv has been written!
